# W2 Lab — Prompting and Reasoning: Chain-of-Thought and Examples

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/stml_2026/blob/main/lectures/week02/W2_lab_prompting.ipynb)

**Goal.** Make a model more accurate by letting it think out loud (chain-of-thought),
teach it format and tone by showing examples (few-shot), and verify each prompt
change by scoring it on a fixed evalset with code.

This week's theory: reasoning accuracy depends on what the prompt makes the model
write before the answer. **Chain-of-thought (CoT)** = instructing the model to write
its intermediate steps first (Wei et al., 2022). **Few-shot prompting** = placing
worked examples in the prompt for the model to imitate.

The path: setup (1) → thinking step by step (2) → a code-graded eval: baseline →
format fix → your CoT prompt, target 11 of 12 (3) → few-shot and the email exercise
(4) → self-consistency (5) → practice drills (6) → completion and submission (7).

*Runtime:* Google Colab, top-to-bottom, ~90 minutes. **This lab is collected** — fill in Section 1.3 and follow Section 7 to submit. Cells marked ✍️ ask for your own writing — a fill-in or a written prediction.

*Sources:* Sections 2 and 4 adapt Anthropic's *Prompt Engineering Interactive
Tutorial*, ch. 6–7 (observation prompts, email dataset, grading loop verbatim);
Section 3, Anthropic's *Prompt Evaluations*, lesson 3 (dataset, prompts, graders
verbatim); Section 5 applies Wang et al. (2022), *Self-Consistency*, to the Section 3
eval — no source lab. Only adaptation: the course API standard, `aisuite` + a pasted
key, no assistant-prefill turns. The Section 6 drills are course-authored in the style of the tutorial exercises.


## 1. Setup

### 1.1 Installation

`aisuite` exposes multiple providers (OpenAI, Anthropic) behind one interface, so the
same code runs whichever provider your key belongs to.

*Do:* run the cell (~30 seconds, once per session).


In [ ]:
%pip install -q "aisuite[openai,anthropic]"

### 1.2 API key and model

An **API key** = the secret string that identifies your account to the provider and
bills usage to it (issuing steps: the API Setup guide on the course site); do not
share the notebook with the key inside.

*Do:* replace `PASTE-YOUR-KEY-HERE` with your key and run the cell.


In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "PASTE-YOUR-KEY-HERE"

MODEL = "openai:gpt-4o-mini"   # Anthropic accounts: MODEL = "anthropic:claude-haiku-4-5" and set ANTHROPIC_API_KEY instead

### 1.3 Submission identity

This lab is collected. The completion check prints these values, and a blank name
fails it.

*Do:* fill in your name and student ID, run the cell.

In [ ]:
STUDENT_NAME = ""
STUDENT_ID = ""

print(f"submitting as: {STUDENT_NAME or '(name missing)'} ({STUDENT_ID or 'ID missing'})")

### 1.4 Client and helpers

Last week every call was written out by hand — build a message list, call `create`,
read `.choices[0].message.content` — fourteen times, and the shape never changed.
This week that shape gets a name. `chat` wraps exactly those lines and counts calls
and tokens; `ask` builds the one-question list (an optional `system` dict, then one
`user` dict) and hands it to `chat`. Temperature defaults to 0.0 so runs are
comparable; Section 5 raises it deliberately.

*Do:* run the cell unchanged — nothing in it is new, it is W1's lines, named.

In [ ]:
import aisuite

client = aisuite.Client()

n_calls = 0
n_prompt_tokens = 0
n_completion_tokens = 0

def chat(messages, temperature=0.0, **kwargs):
    """Message list -> assistant reply text. Extra kwargs pass through to the provider."""
    global n_calls, n_prompt_tokens, n_completion_tokens
    response = client.chat.completions.create(
        model=MODEL, messages=messages, temperature=temperature, **kwargs)
    n_calls += 1
    usage = getattr(response, "usage", None)
    if usage is not None:
        n_prompt_tokens += usage.prompt_tokens
        n_completion_tokens += usage.completion_tokens
    return response.choices[0].message.content

def ask(prompt, system=None, temperature=0.0, **kwargs):
    """Single question (optional system instruction) -> reply text."""
    messages = ([{"role": "system", "content": system}] if system else [])
    messages.append({"role": "user", "content": prompt})
    return chat(messages, temperature=temperature, **kwargs)

### 1.5 Verification

*Do:* run the cell and confirm the output is exactly `ready` — the same call that
opened W1, one line instead of four.

In [ ]:
print(ask("Reply with exactly: ready"))

Any error here is a setup problem, not a code problem — recheck the API Setup guide
before continuing.


## 2. Thinking Step by Step

Thinking out loud sometimes makes the model more accurate, particularly on tasks with
a trap or a constraint — and thinking only counts when written out: a model told to "think
silently and output only the answer" does no thinking at all (source: tutorial ch. 6).

### 2.1 A sentiment that flips

The praise in the review below comes from someone living under a rock since 1900;
asked directly, the model takes "unrelated" literally.

*Do:* run the cell and read the verdict.


In [ ]:
## Asked directly
REVIEW_PROMPT = """Is this movie review sentiment positive or negative?

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since the year 1900."""

print(ask(REVIEW_PROMPT))

The source's fix: spell out the thinking — write the best arguments for each side in
tags, then answer.

*Do:* run the cell and compare verdicts.


In [ ]:
## Think first, then answer
THINK_FIRST_PROMPT = """Is this review sentiment positive or negative? First, write the best arguments for each side in <positive-argument> and <negative-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since 1900."""

print(ask(THINK_FIRST_PROMPT, system="You are a savvy reader of movie reviews."))

Writing out both sides let the model notice what the second sentence does to the
first.

### 2.2 Recall under a constraint

The same device on a factual question the model tends to miss when answering cold.

*Do:* run both cells and compare the answers.


In [ ]:
## Asked cold
print(ask("Name a famous movie starring an actor who was born in the year 1956."))

In [ ]:
## Brainstorm first, then answer
print(ask("Name a famous movie starring an actor who was born in the year 1956. "
          "First brainstorm about some actors and their birth years in <brainstorm> tags, "
          "then give your answer."))

Two anecdotes are not evidence. Whether step-by-step thinking helps in general — and
by how much — is a measurement question; the next section builds the instrument.


## 3. A Code-Graded Evaluation

**Code-graded evaluation** = scoring a prompt by running it over a fixed test set
with known answers and grading the outputs with code (source: *Prompt Evaluations*,
lesson 3; dataset and prompts verbatim). The task: from a statement about an animal,
answer how many legs it has; several statements are deliberately tricky (a fox that
lost a leg and regrew two).

### 3.1 Eval set and first prompt

*Do:* run the two cells and read the outputs.


In [ ]:
eval_data = [
    {"animal_statement": "The animal is a human.", "golden_answer": "2"},
    {"animal_statement": "The animal is a snake.", "golden_answer": "0"},
    {"animal_statement": "The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.", "golden_answer": "5"},
    {"animal_statement": "The animal is a dog.", "golden_answer": "4"},
    {"animal_statement": "The animal is a cat with two extra legs.", "golden_answer": "6"},
    {"animal_statement": "The animal is an elephant.", "golden_answer": "4"},
    {"animal_statement": "The animal is a bird.", "golden_answer": "2"},
    {"animal_statement": "The animal is a fish.", "golden_answer": "0"},
    {"animal_statement": "The animal is a spider with two extra legs", "golden_answer": "10"},
    {"animal_statement": "The animal is an octopus.", "golden_answer": "8"},
    {"animal_statement": "The animal is an octopus that lost two legs and then regrew three legs.", "golden_answer": "9"},
    {"animal_statement": "The animal is a two-headed, eight-legged mythical creature.", "golden_answer": "8"},
]
print(len(eval_data), "items")

In [ ]:
def build_input_prompt(animal_statement):
    return f"""You will be provided a statement about an animal and your job is to determine how many legs that animal has.

Here is the animal statement.
<animal_statement>{animal_statement}</animal_statement>

How many legs does the animal have? Please respond with a number"""

def grade_completion(output, golden_answer):
    return output.strip() == golden_answer

outputs = [ask(build_input_prompt(q["animal_statement"])) for q in eval_data]
for output, q in zip(outputs, eval_data):
    print(f"golden={q['golden_answer']:>2}  output={output!r}")
score_v1 = sum(grade_completion(o, q["golden_answer"]) for o, q in zip(outputs, eval_data))
print(f"\nscore: {score_v1}/12")

Two separate problems: sentences instead of bare numbers (a formatting failure — the
right value graded wrong), and wrong numbers on the tricky statements (a reasoning
failure). The next two prompts fix them one at a time.

### 3.2 Fixing the format

One added line: *"Respond only with a numeric digit, like 2 or 6, and nothing else."*

*Do:* run the cell; formatting failures should disappear, tricky items should still
miss.


In [ ]:
def build_input_prompt2(animal_statement):
    return f"""You will be provided a statement about an animal and your job is to determine how many legs that animal has.

Here is the animal statement.
<animal_statement>{animal_statement}</animal_statement>

How many legs does the animal have? Respond only with a numeric digit, like 2 or 6, and nothing else."""

outputs2 = [ask(build_input_prompt2(q["animal_statement"])) for q in eval_data]
score_v2 = sum(grade_completion(o, q["golden_answer"]) for o, q in zip(outputs2, eval_data))
for output, q in zip(outputs2, eval_data):
    mark = "PASS" if grade_completion(output, q["golden_answer"]) else "FAIL"
    print(f"{mark}  golden={q['golden_answer']:>2}  output={output!r}")
print(f"\nscore: {score_v2}/12")

### 3.3 Chain of thought, graded ✍️ (core)

The remaining misses are reasoning failures; Section 2's device is the candidate fix.

Write `build_input_prompt3`: keep the task statement; instruct the model to reason
step by step inside `<thinking>` tags, then put the final answer — just the integer —
inside `<answer>` tags. `extract_answer` (source verbatim) pulls the number out for
grading. Target: **at least 11 of 12**.

*Do:* fill in the prompt, run the check cell, iterate until the target is reached.


In [ ]:
### FILL IN (START) ###
def build_input_prompt3(animal_statement):
    return f"""You will be provided a statement about an animal and your job is to determine how many legs that animal has.

Here is the animal statement.
<animal_statement>{animal_statement}</animal_statement>

How many legs does the animal have?"""
### FILL IN (END) ###

In [ ]:
import re

def extract_answer(text):
    match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL)
    return match.group(1).strip() if match else None

outputs3 = [ask(build_input_prompt3(q["animal_statement"])) for q in eval_data]
extracted = [extract_answer(o) for o in outputs3]
cot_score = sum(e == q["golden_answer"] for e, q in zip(extracted, eval_data))
for e, q in zip(extracted, eval_data):
    mark = "PASS" if e == q["golden_answer"] else "FAIL"
    print(f"{mark}  golden={q['golden_answer']:>2}  extracted={e}  {q['animal_statement'][:58]}")
print(f"\nscore: {cot_score}/12   (target: >= 11)")

The source run reached 12/12 with this prompt shape. The eval made a prompting claim
— "thinking step by step helps here" — checkable, and the same three-step sequence
(baseline → format fix → reasoning fix, each scored) is how prompts are improved in
practice.


## 4. Using Examples (Few-Shot)

**Few-shot prompting** = placing example question–answer pairs in the prompt; the
model imitates their format, tone, and procedure. "Zero-shot" / "one-shot" / "n-shot"
counts the examples. Often it is easier to show than to describe (source: tutorial
ch. 7).

### 4.1 The parent bot

A bot for children's questions; asked cold, the model answers like an encyclopedia.

*Do:* run both cells and compare the tone.


In [ ]:
## Asked cold
print(ask("Will Santa bring me presents on Christmas?"))

In [ ]:
## One worked example in the prompt
PARENT_BOT_PROMPT = """Please complete the conversation by writing the next line, speaking as "A".
Q: Is the tooth fairy real?
A: Of course, sweetie. Wrap up your tooth and put it under your pillow tonight. There might be something waiting for you in the morning.
Q: Will Santa bring me presents on Christmas?"""

print(ask(PARENT_BOT_PROMPT))

One example fixed tone and length at once — no instruction described either.

### 4.2 Email classification by examples ✍️ (tutorial exercise 7.1)

Classify customer emails into

- (A) Pre-sale question
- (B) Broken or defective item
- (C) Billing question
- (D) Other (please explain)

The grader checks the **last character** of the output against the correct letter.
Rewrite `PROMPT` so that few-shot examples of emails plus correctly formatted answers
make every output end with the right letter. (Source solution: each example answer
ends "The correct category is: X"; put the examples in the user prompt — the source's
assistant-prefill turn is Anthropic-specific.) Target: **4 of 4**.

*Do:* rewrite `PROMPT`, run the cell, and iterate until all four emails read `PASS`.


In [ ]:
import re

### FILL IN (START) ###
PROMPT = """Please classify this email as either green or blue: {email}"""
### FILL IN (END) ###

EMAILS = [
    "Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.",  # (B) Broken or defective item
    "Can I use my Mixmaster 4000 to mix paint, or is it only meant for mixing food?",  # (A) Pre-sale question OR (D) Other (please explain)
    "I HAVE BEEN WAITING 4 MONTHS FOR MY MONTHLY CHARGES TO END AFTER CANCELLING!!  WTF IS GOING ON???",  # (C) Billing question
    "How did I get here I am not good with computer.  Halp.",  # (D) Other (please explain)
]
ANSWERS = [["B"], ["A", "D"], ["C"], ["D"]]

email_score = 0
for i, email in enumerate(EMAILS):
    response = ask(PROMPT.format(email=email))
    grade = any(bool(re.search(ans, response[-1])) for ans in ANSWERS[i])
    email_score += grade
    print(f"{'PASS' if grade else 'FAIL'}  expected {'/'.join(ANSWERS[i])}  got: ...{response[-40:]!r}")
print(f"\nscore: {email_score}/4")

The source's chapter 6 solves this task with instructions alone; examples reach the
same format with less instruction-writing, and the two techniques compose — worked
examples *of step-by-step solutions* are the few-shot CoT exemplars of Wei et al.
(2022) (notes Ch. 2).


## 5. Self-Consistency

**Self-consistency** = sample several reasoning paths for the same question at
nonzero temperature, then take the majority of the extracted answers: wrong paths
scatter, correct paths agree (procedure of Wang et al., 2022, applied to the
Section 3 eval — no source lab).

*Do:* run the cell — five samples of the 3.3 prompt on the trickiest statement at
`t=1.0` — and compare the samples with the majority.


In [ ]:
from collections import Counter

HARD = eval_data[2]   # the fox that lost a leg and regrew two

samples = [extract_answer(ask(build_input_prompt3(HARD["animal_statement"]), temperature=1.0))
           for _ in range(5)]
votes = [s for s in samples if s is not None]
majority = Counter(votes).most_common(1)[0][0] if votes else None

print("samples :", samples)
print("majority:", majority, "  golden:", HARD["golden_answer"])

A single sampled run stands or falls with one path; the vote marginalizes over paths.
The price is five calls instead of one — the accuracy-for-compute exchange named
test-time compute (notes Ch. 2), which Ch. 11's theory follows into the model's own training.

## 6. Practice Drills ✍️

Each drill hands you a prompt that fails its check. The surrounding code stays
fixed; only the prompt string — or the sampling you add — changes, until the check
prints `PASS`. All three drills are part of the submission.

6.1 exercises the instruction form of CoT (notes §2.3, Method 1); 6.2 exercises the
exemplar form (Method 2) — a format no instruction fully states; 6.3 turns Section 5
from a demonstration into your own code.

### 6.1 An instruction that elicits the solution ✍️

`COT_INSTRUCTION` is appended after the word problem. The starter is empty, so the
model answers in whatever shape it likes and the extraction finds no `ANSWER:` line.

*Do:* write the instruction, run, iterate until the line reads `PASS`.

Hints: demand the solution step by step, and state the exact final line —
`ANSWER: <number>`. The check extracts that line and compares the number.

In [ ]:
LIBRARY_PROBLEM = ("A library has 4 shelves with 38 books each. During the day 47 books "
                   "are checked out and 26 are returned. How many books are on the shelves now?")

### FILL IN (START) ###
COT_INSTRUCTION = ""
### FILL IN (END) ###

output = ask(LIBRARY_PROBLEM + "\n" + COT_INSTRUCTION)
match = re.search(r"ANSWER:\s*(\d+)", output)
drill_a = match is not None and match.group(1) == "131"

print(output)
print(f"\n{'PASS' if drill_a else 'FAIL'}  extracted: {match.group(1) if match else None}   expected: 131")

### 6.2 An exemplar that pins the format ✍️

The check accepts exactly one output shape: `DATE: 2026-03-10` — nothing before it,
nothing after it. Instructions tend to leave something loose (a trailing period, a
sentence around the date); a worked exemplar states the shape by showing it.

*Do:* add one or two worked note → answer exemplars inside `DATE_PROMPT`, above the
test note, then run; iterate until `PASS`.

Hints: format each exemplar exactly as the output should look — for instance a note
about June 9th, 2026 answered with `DATE: 2026-06-09` on its own line — and keep the
test note last, laid out the same way as the exemplars.

In [ ]:
TEST_NOTE = "Team offsite moved from March 3rd to March 10th, 2026."

### FILL IN (START) ###
DATE_PROMPT = """Extract the final date from the note.

Note: {note}"""
### FILL IN (END) ###

output = ask(DATE_PROMPT.format(note=TEST_NOTE))
drill_b = output.strip() == "DATE: 2026-03-10"

print(repr(output))
print(f"\n{'PASS' if drill_b else 'FAIL'}  expected exactly 'DATE: 2026-03-10'")

### 6.3 The vote, written by you ✍️

Section 5 demonstrated self-consistency with given code. Here the statement is new —
a spider that lost three legs, golden answer 5 — and the sampling is yours.

*Do:* fill in `samples`: five runs of your 3.3 prompt on `TRICKY` at
`temperature=1.0`, each passed through `extract_answer`. The vote below is already
written. Iterate until `PASS`.

Hints: the pattern is Section 5's cell — a list comprehension over `range(5)`. Keep
`temperature=1.0`; at 0.0 the five samples are five copies of a single path.

In [ ]:
TRICKY = {"animal_statement": "The animal is a spider that lost three legs.",
          "golden_answer": "5"}

### FILL IN (START) ###
samples = []
### FILL IN (END) ###

votes = [s for s in samples if s is not None]
majority = Counter(votes).most_common(1)[0][0] if votes else None
drill_c = majority == TRICKY["golden_answer"]

print("samples :", samples)
print(f"{'PASS' if drill_c else 'FAIL'}  majority: {majority}   golden: {TRICKY['golden_answer']}")

## 7. Completion and Submission

This lab is collected. Completion criteria: the identity line is filled, the CoT
prompt reaches the Section 3 target, the few-shot prompt classifies every email, and
all three drills pass. Grading checks these structural facts, never prose quality.

*Do:* run the notebook top to bottom once more so every output is saved, confirm
every row below reads `PASS`, then download the notebook (**File → Download →
Download .ipynb**) and submit it the way announced in class.

In [ ]:
completion = {
    "name and student ID filled in (1.3)":       bool(STUDENT_NAME.strip()) and bool(STUDENT_ID.strip()),
    "CoT prompt reaches >= 11/12 (3.3)":         cot_score >= 11,
    "few-shot emails all pass (4.2)":            email_score == 4,
    "drill 6.1 — instruction elicits ANSWER":    drill_a,
    "drill 6.2 — exemplar pins the format":      drill_b,
    "drill 6.3 — the vote finds the answer":     drill_c,
}
print(f"submitted by: {STUDENT_NAME or '?'} ({STUDENT_ID or '?'})\n")
for item, ok in completion.items():
    print(f"{'PASS' if ok else 'FAIL':4}  {item}")
print("\nLAB COMPLETE — download and submit" if all(completion.values()) else "\nNOT COMPLETE YET")

---

Reference fill-ins: `labs/checkpoints/week02/solution.py` (lab and homework
together), published after the homework deadline.

W3 turns plain Python functions into tools the model can call (DeepLearning.AI,
*Agentic AI*, Module 3): function calling, the request–execute–reinject trace, and a
measured routing score — on the same `chat` helper.
